# ECG specialist — PULSE-7B

Generate **ECG reports only** from the rendered query ECG plus the identical frozen retrieval contexts.
PULSE is an ECG-image model, not a raw-waveform model. Use the official AIMedLab/PULSE Python 3.10 environment and set `PULSE_REPO_ROOT` (default: the repository's ignored `external/PULSE` directory). Image tiles count toward the 4096-token context. `E4_PULSE_COMPACT_COMPONENT_CAP=120` sets the fixed evidence excerpt budget for the revised PULSE-only runner.

The checkpoint discloses MIMIC-IV-ECG-derived training data. Its results test within-domain RAG augmentation;
they must not be described as uncontaminated external validation. Output is append-only and resumable.

## Compact PULSE protocol
Use the Python 3.10 PULSE kernel, not the Unsloth or Qwen kernel. Upload
`e4_pulse_generation.py` with this notebook; existing shared model runners are unchanged.
This new protocol requests a plain ECG report, NOT JSON or an evidence trace. Structured-output
and attribution metrics are not directly comparable with the other models' JSON prompts.
All conditions use the same compact instructions and a fixed 120-character component cap.
Lab names/percentiles are taken from observed structured entries; no clinical abnormality is inferred.
All selected anchors and ECG image tiles are retained. The entire cohort is checked against the
4096-token context, reserving 512 output tokens, before any generation attempt is written.
If preflight fails, inspect `pulse7b_ecg_budget_preflight.json`; do not raise the model context limit.
First run the 50-cell pilot, inspect actual reports and schema-echo rejection, then set
`E4_MAX_SPECIALIST_ROWS=0` to continue. Restarting skips both successes and failed attempts.
Keep `pulse_compact_v1` and all settings fixed when resuming. Any code/settings changes require a new name.
Do not reuse old PULSE successes or copy model files between run directories. Final cross-model evaluation
requires explicit model-to-run selection; the current evaluator reads one run directory at a time.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
# Fixed name for this revised PULSE protocol. Keep it unchanged when resuming.
os.environ['E4_GENERATION_RUN'] = 'pulse_compact_v1'
from e4_generation_runtime import generation_root
print('PULSE output:', generation_root(EXPERIMENT4_ROOT))

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi, get_token, login

MODEL_REPO = 'PULSE-ECG/PULSE-7B'
token_file_value = os.environ.get('HF_TOKEN_FILE')
HF_TOKEN_FILE = Path(token_file_value).expanduser() if token_file_value else None
hf_token = HF_TOKEN_FILE.read_text().strip() if HF_TOKEN_FILE is not None and HF_TOKEN_FILE.is_file() else None
if hf_token and any(char.isspace() for char in hf_token):
    raise ValueError(f'Hugging Face token file must contain exactly one token: {HF_TOKEN_FILE}')
hf_token = (hf_token or os.environ.get('HF_TOKEN') or
            os.environ.get('HUGGING_FACE_HUB_TOKEN') or get_token())
if not hf_token:
    hf_token = getpass('Enter a Hugging Face read token (input is hidden): ').strip()
login(token=hf_token, add_to_git_credential=False)
info = HfApi().model_info(MODEL_REPO, token=hf_token)
print({'model_repo': MODEL_REPO, 'revision': info.sha,
       'token_source': str(HF_TOKEN_FILE) if HF_TOKEN_FILE is not None and HF_TOKEN_FILE.is_file() else 'environment/cache/prompt'})

In [ ]:
from e4_ecg_specialists import validate_specialist_environment
validate_specialist_environment('pulse7b_ecg')
import google.protobuf; print("protobuf:", google.protobuf.__version__)

In [ ]:
import os
from e4_pulse_generation import run_pulse
protocol = run_pulse(
    'pulse7b_ecg',
    max_new_tokens=int(os.environ.get('E4_PULSE_MAX_NEW_TOKENS', '512')),
    max_input_tokens=int(os.environ.get('E4_PULSE_MAX_INPUT_TOKENS', '3584')),
    max_rows=int(os.environ.get('E4_MAX_SPECIALIST_ROWS', '50')),
    component_cap=int(os.environ.get('E4_PULSE_COMPACT_COMPONENT_CAP', '120')),
)
protocol